# 🚀 Image Captioning — GPU Model Trainer (Google Colab)

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook will:
1. Verify GPU availability
2. Clone your project from GitHub
3. Install dependencies
4. Mount Google Drive to persist trained model
5. Run the full training pipeline with EarlyStopping & checkpoints
6. Plot training history

## ✅ Step 1 — Verify GPU

In [10]:
!nvidia-smi

Wed Jul  8 15:05:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'Num GPUs Available: {len(gpus)}')
for gpu in gpus:
    print(f'  → {gpu}')

if not gpus:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → GPU')

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('\n✅ GPU ready!')

Num GPUs Available: 1
  → PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✅ GPU ready!


## 📁 Step 2 — Mount Google Drive

In [11]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/imcaption_full'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'📂 Model will be saved to: {DRIVE_SAVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Model will be saved to: /content/drive/MyDrive/imcaption_full


## 📦 Step 3 — Clone Repo & Install Dependencies

> 💡 Replace the `REPO_URL` with your actual GitHub repository URL.

In [12]:
import os

REPO_URL = 'https://github.com/divyanshrajput118/imcaption.git' 

if not os.path.exists('/content/imcaption'):
    os.system(f'git clone {REPO_URL} /content/imcaption')
else:
    print('Repo already cloned. Pulling latest...')
    os.system('cd /content/imcaption && git pull')

os.chdir('/content/imcaption')
print(f'Working directory: {os.getcwd()}')

Repo already cloned. Pulling latest...
Working directory: /content/imcaption


In [6]:
!git pull

Already up to date.


In [13]:
!pip install -e . -q
!pip install -r requirements_colab.txt -q
print('✅ Dependencies installed!')

  Preparing metadata (setup.py) ... done
✅ Dependencies installed!


## 📥 Step 4 — Check / Load Artifacts

> ⚡ If you have artifacts saved in Drive from a previous run, set `COPY_FROM_DRIVE = True`.

In [ ]:
!python main.py

[2026-07-08 15:06:43,068: INFO: utils: NumExpr defaulting to 2 threads.]
[2026-07-08 15:06:47,463: INFO: main: >>>>>> stage Data_Ingestion_Stage started <<<<<<]
[2026-07-08 15:06:47,465: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-07-08 15:06:47,467: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-08 15:06:47,467: INFO: common: created directory at: artifacts]
[2026-07-08 15:06:47,467: INFO: common: created directory at: artifacts/data_ingestion]
[2026-07-08 15:06:47,468: INFO: data_ingestion: Downloading data from https://drive.google.com/file/d/1X-l4QnuhIYMXm-Lgl2N0MO-xR75ly3OD/view?usp=sharing into file artifacts/data_ingestion/data.zip]
Downloading...
From (original): https://drive.google.com/uc?id=1X-l4QnuhIYMXm-Lgl2N0MO-xR75ly3OD
From (redirected): https://drive.google.com/uc?id=1X-l4QnuhIYMXm-Lgl2N0MO-xR75ly3OD&confirm=t&uuid=352aeef5-75fa-4154-b3c8-6ec48bd91e52
To: /content/imcaption/artifacts/data_ingestion/data.zip
100% 1.11G/1.11

Extracting features:  47% 96/203 [00:34<00:29,  3.62batch/s]^C


In [ ]:
# Run this immediately after !python main.py completes
import shutil, os

# Save the trained model to Drive
shutil.copy(
    '/content/imcaption/artifacts/training/model.keras',
    '/content/drive/MyDrive/imcaption_trained/model.keras'
)
print("✅ Trained model saved to Drive!")


In [15]:
import os

required = [
    'artifacts/data_ingestion/Images',
    'artifacts/data_transformation/vectorizer_data.pkl',
    'artifacts/data_transformation/train_images_captions.pkl',
    'artifacts/prepare_base_model/feature_extractor.keras',
    'artifacts/prepare_base_model/main_model.keras',
]

missing = [p for p in required if not os.path.exists(p)]

if missing:
    print('⚠️  Missing artifacts:')
    for m in missing:
        print(f'  ✗ {m}')
    print('\nEither set COPY_FROM_DRIVE=True below or run the full pipeline.')
else:
    print('✅ All required artifacts found!')